<a href="https://colab.research.google.com/github/EsarFatima/MachineLearning-flyrank-/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/EsarFatima/MachineLearning-flyrank-"
REPO_DIR = "MachineLearning-flyrank-"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(df.shape[0], "pages | base decline rate:", round(df["is_declining_label"].mean(), 3))
print()

# Rate/volume columns this audit tests below -- percentiles show the shape before deciding anything.
for c in ["ctr", "engagement_rate", "scroll_rate", "ai_traffic_pct", "impressions_90d"]:
    d = df[c].describe(percentiles=[.5, .9, .99])
    print(f"{c:16s} median={d['50%']:.2f}  p90={d['90%']:.2f}  p99={d['99%']:.2f}  max={d['max']:.2f}")

print()
print("zero-mass check (the median hides this):")
print("  ctr == 0:            ", round((df['ctr']==0).mean(), 3))
print("  engagement_rate == 0:", round((df['engagement_rate']==0).mean(), 3))
print("  scroll_rate == 0:    ", round((df['scroll_rate']==0).mean(), 3))

30000 pages | base decline rate: 0.542

ctr              median=0.07  p90=0.65  p99=8.33  max=100.00
engagement_rate  median=0.00  p90=6.94  p99=33.33  max=100.00
scroll_rate      median=5.00  p90=50.00  p99=100.00  max=300.00
ai_traffic_pct   median=0.00  p90=0.00  p99=16.67  max=300.00
impressions_90d  median=731.00  p90=12136.40  p99=73505.83  max=517715.00

zero-mass check (the median hides this):
  ctr == 0:             0.44
  engagement_rate == 0: 0.721
  scroll_rate == 0:     0.374


**What the shape says, before any test.** Every rate column here is heavy-tailed with a hard floor at zero, and that floor is a real, large population, not a rounding artifact: 44.0% of pages have `ctr == 0` (zero clicks in 90 days) and 72.1% have `engagement_rate == 0`. That means the median of each column sits at or near zero, so a plain correlation on raw values would be dominated by the long tail above p90 (ctr's p99 is 8.33 against a median of 0.07). Per the skill card I bucket instead of correlating, and I always keep the zero-population as its own bucket rather than folding it into "low" — a page with literally no clicks is a different story from a page with a low-but-nonzero CTR, and merging them would hide that distinction below. `impressions_90d` gets the same heavy-tail treatment (already `log1p`'d for modeling in the w03 leakage notebook and w05).


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [ ]:
# Signal test #1 -- claim: "worse-ranked pages are more likely currently declining."
order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
sig1 = (
    df.groupby("position_tier")["is_declining_label"]
      .agg(decline_rate="mean", n="count")
      .reindex(order)
)
sig1["decline_rate"] = sig1["decline_rate"].round(3)
print(sig1)
print("base rate:", round(df["is_declining_label"].mean(), 3))

               decline_rate      n
position_tier                     
top_3                 0.241   2321
page_1                0.570  11814
striking              0.610   7304
page_3_5              0.562   7242
deep                  0.344   1319
base rate: 0.542


**Verdict — MIXED.** All five buckets clear the sample floor easily (smallest is `deep` at n=1,319). But the shape is a hump, not a line: `top_3` (0.241) and `deep` (0.344) both sit well *below* the base rate, while the three middle tiers (`page_1` 0.570, `striking` 0.610, `page_3_5` 0.562) all sit *above* it. Reading it plainly — pages ranking in the top 3 are stable winners, and pages already buried past position 50 have little visibility left to lose; it's the pages ranking respectably-but-not-great that are actually "in motion." Same hump shape the w04 baseline found for impression volume — rank and traffic size both turn out to be U-shaped risk signals here, not straight lines.

In [ ]:
# Signal test #2 -- claim: "lower-CTR pages are more likely currently declining."
ctr_tier = pd.Series("zero", index=df.index)
nz = df["ctr"] > 0
ctr_tier.loc[nz] = pd.qcut(df.loc[nz, "ctr"], q=3, duplicates="drop").astype(str)
df["ctr_tier"] = ctr_tier

sig2 = df.groupby("ctr_tier")["is_declining_label"].agg(decline_rate="mean", n="count")
sig2 = sig2.reindex(["zero"] + [c for c in sig2.index if c != "zero"])
sig2["decline_rate"] = sig2["decline_rate"].round(3)
print(sig2)
print("base rate:", round(df["is_declining_label"].mean(), 3))

                              decline_rate      n
ctr_tier                                         
zero                                 0.497  13212
(0.009000000000000001, 0.16]         0.653   5714
(0.16, 0.39]                         0.577   5601
(0.39, 100.0]                        0.500   5473
base rate: 0.542


**Verdict — MIXED.** Among pages that get *any* clicks, the claim holds cleanly and monotonically: decline rate falls from 0.653 (lowest nonzero-CTR third) to 0.577 to 0.500 (highest third) as CTR rises — all three buckets comfortably above the floor (n ≥ 5,473). But the `zero` bucket — 44.0% of the whole dataset — breaks the line: it sits at 0.497, *below* the lowest nonzero third, not above it. A page with zero clicks isn't "worse than low-CTR," it's a different population — likely one with too little exposure for CTR to mean anything yet, the same reasoning as the low-impression pages in the w04 audit. So: a real signal among pages with measurable clicks, no signal (or a different mechanism) for the zero-click quarter.

In [ ]:
# Signal test #3 -- claim: "lower-engagement pages are more likely currently declining."
eng_tier = pd.Series("zero", index=df.index)
nz2 = df["engagement_rate"] > 0
eng_tier.loc[nz2] = pd.qcut(df.loc[nz2, "engagement_rate"], q=3, duplicates="drop").astype(str)
df["eng_tier"] = eng_tier

sig3 = df.groupby("eng_tier")["is_declining_label"].agg(decline_rate="mean", n="count")
sig3 = sig3.reindex(["zero"] + [c for c in sig3.index if c != "zero"])
sig3["decline_rate"] = sig3["decline_rate"].round(3)
print(sig3)
print("base rate:", round(df["is_declining_label"].mean(), 3))

               decline_rate      n
eng_tier                          
zero                  0.544  21629
(0.049, 3.13]         0.548   2834
(3.13, 7.46]          0.536   2746
(7.46, 100.0]         0.525   2791
base rate: 0.542


**Verdict — FALSE.** All four buckets sit within about two points of the 0.542 base rate (0.525 to 0.548), with no consistent direction as engagement rises. Every bucket clears the floor easily, so this isn't a sample-size illusion — `engagement_rate` on its own, at this grain, just doesn't separate declining pages from stable ones. That's a real, publishable answer: not every plausible-sounding signal shows up in the data, and this one doesn't.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [ ]:
# Flag-linked test -- FlyRank's needs_ctr_fix logic assumes: "a page that ranks well but gets
# comparatively few clicks for that rank is underperforming and worth fixing." Test it directly
# by restricting to well-positioned pages, then repeating the CTR cut inside that slice only.
good_pos = df["position_tier"].isin(["top_3", "page_1"])
sub = df[good_pos].copy()

ctr_tier2 = pd.Series("zero", index=sub.index)
nz3 = sub["ctr"] > 0
ctr_tier2.loc[nz3] = pd.qcut(sub.loc[nz3, "ctr"], q=3, duplicates="drop").astype(str)
sub["ctr_tier"] = ctr_tier2

sig4 = sub.groupby("ctr_tier")["is_declining_label"].agg(decline_rate="mean", n="count")
sig4 = sig4.reindex(["zero"] + [c for c in sig4.index if c != "zero"])
sig4["decline_rate"] = sig4["decline_rate"].round(3)
print(sig4)
print("base rate, well-positioned pages only:", round(sub["is_declining_label"].mean(), 3))
print("base rate, whole dataset:             ", round(df["is_declining_label"].mean(), 3))
print("n well-positioned:", good_pos.sum())

                             decline_rate     n
ctr_tier                                       
zero                                0.442  5820
(0.009000000000000001, 0.2]         0.643  2802
(0.2, 0.48]                         0.579  2802
(0.48, 100.0]                       0.475  2711
base rate, well-positioned pages only: 0.516
base rate, whole dataset:              0.542
n well-positioned: 14135


**Verdict — CONFIRMED (with one caveat).** Restricted to well-positioned pages only (`top_3`/`page_1`, n=14,135), the needs_ctr_fix assumption holds up cleanly and monotonically: decline rate rises from 0.475 (top nonzero-CTR third) to 0.579 to 0.643 (bottom nonzero-CTR third) as CTR falls — every bucket well above the floor (smallest n=2,711). That's a specific finding, not just "low CTR is bad": it's "low CTR *given the page is already ranking well* is bad," which is exactly the story needs_ctr_fix tells — a title/snippet problem, not a ranking problem. The caveat is the same shape as Signal test #2: the `zero`-CTR slice (n=5,820, still well-positioned) sits at 0.442 — the *lowest* of the four, not the highest. Zero clicks despite a good rank reads more like a brand-new page or a tracking gap than a decaying one, so I'd keep needs_ctr_fix's low-but-nonzero-CTR framing and flag its zero-click edge case as a separate, unverified assumption rather than folding it into the same rule.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [ ]:
# Recap for the content team -- one row per verdict.
summary = pd.DataFrame([
    {"signal": "position_tier (rank)", "verdict": "MIXED",
     "shape": "hump -- worst risk mid-rank"},
    {"signal": "ctr (nonzero)", "verdict": "MIXED",
     "shape": "monotonic -- worst risk = low CTR"},
    {"signal": "engagement_rate", "verdict": "FALSE",
     "shape": "flat -- no separation from base rate"},
    {"signal": "ctr within well-positioned (needs_ctr_fix)", "verdict": "CONFIRMED",
     "shape": "monotonic -- low CTR + good rank = highest risk"},
])
print(summary.to_string(index=False))

                                    signal   verdict                                           shape
                      position_tier (rank)     MIXED                     hump -- worst risk mid-rank
                             ctr (nonzero)     MIXED               monotonic -- worst risk = low CTR
                           engagement_rate     FALSE            flat -- no separation from base rate
ctr within well-positioned (needs_ctr_fix) CONFIRMED monotonic -- low CTR + good rank = highest risk


**What a content team should take from this.** The one signal that's both real *and* matches an existing product flag is CTR-within-good-rank — `needs_ctr_fix` is picking up something genuinely observed in this slice, so a review queue built on it isn't chasing noise. Rank and impression volume (this audit and w04's baseline agree) are both hump-shaped, not linear — a "the worse the metric, the higher the risk" rule would misfire at both extremes, so any scoring rule should treat "moderate-but-not-terrible" as its own risk band rather than a threshold. And `engagement_rate` alone isn't worth building a rule around at this grain — that's a directional, decision-support finding, not a data gap to explain away.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.